In [1]:
from __future__ import annotations

import logging
import math
from typing import Callable, Dict, List, Literal, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

from qsimbench import get_outcomes, get_index

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------

logger = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Global configuration
# ---------------------------------------------------------------------------

# Global RNG seed used for qsimbench calls (for reproducibility).
SEED: int = 42

# Number of shots used to build the high-shot reference distributions when
# use_final_as_reference == False.
REFERENCE_SHOTS: int = 20_000

# Type alias for outcome-count dictionaries returned by qsimbench.
FreqCounts = Mapping[str, int]

# Distance names we support
DistanceName = Literal["tvd", "hellinger", "js"]
DistanceFn = Callable[[FreqCounts, FreqCounts], float]


# ---------------------------------------------------------------------------
# Basic probability utilities
# ---------------------------------------------------------------------------

def normalised_frequency(frequency: FreqCounts) -> Dict[str, float]:
    """
    Convert frequency counts into a normalized probability distribution.
    """
    total = sum(frequency.values())
    if total == 0:
        return {k: 0.0 for k in frequency}
    inv_total = 1.0 / float(total)
    return {k: v * inv_total for k, v in frequency.items()}


def total_variation_distance(freq1: FreqCounts, freq2: FreqCounts) -> float:
    """
    TVD(p, q) = 0.5 * sum_x |p(x) - q(x)|
    """
    keys = set(freq1.keys()).union(freq2.keys())
    norm1 = normalised_frequency(freq1)
    norm2 = normalised_frequency(freq2)
    return 0.5 * sum(abs(norm1.get(k, 0.0) - norm2.get(k, 0.0)) for k in keys)


def hellinger_distance(freq1: FreqCounts, freq2: FreqCounts) -> float:
    """
    Hellinger distance between two discrete distributions (0..1).

        H(P, Q) = (1/sqrt(2)) * sqrt( sum_x (sqrt(p_x) - sqrt(q_x))^2 )
    """
    p_norm = normalised_frequency(freq1)
    q_norm = normalised_frequency(freq2)
    keys = set(p_norm.keys()) | set(q_norm.keys())
    sq_sum = 0.0
    for k in keys:
        sp = math.sqrt(p_norm.get(k, 0.0))
        sq = math.sqrt(q_norm.get(k, 0.0))
        sq_sum += (sp - sq) ** 2
    h = (1.0 / math.sqrt(2.0)) * math.sqrt(sq_sum)
    logger.debug("hellinger_distance: keys=%d, H=%.6f", len(keys), h)
    return h


def js_distance(freq1: FreqCounts, freq2: FreqCounts) -> float:
    """
    Jensen–Shannon distance between two discrete distributions.

    Uses log base 2 so the divergence is in [0, 1], and returns sqrt(JS divergence),
    which is a metric in [0, 1].
    """
    p_norm = normalised_frequency(freq1)
    q_norm = normalised_frequency(freq2)
    keys = set(p_norm.keys()) | set(q_norm.keys())

    # mixture
    m = {k: 0.5 * (p_norm.get(k, 0.0) + q_norm.get(k, 0.0)) for k in keys}

    def _kl_div(a: Dict[str, float], b: Dict[str, float]) -> float:
        div = 0.0
        for k in keys:
            ak = a.get(k, 0.0)
            if ak <= 0.0:
                continue  # 0 * log(0/.) := 0
            bk = b.get(k, 0.0)
            # In JS, if ak > 0 then bk = (ak + other)/2 > 0, so this should not happen,
            # but we guard anyway for safety.
            if bk <= 0.0:
                continue
            div += ak * math.log(ak / bk, 2.0)
        return div

    js_div = 0.5 * _kl_div(p_norm, m) + 0.5 * _kl_div(q_norm, m)

    # Numerical safety: JS divergence is theoretically >= 0
    if js_div < 0.0 and js_div > -1e-15:
        js_div = 0.0

    d = math.sqrt(max(js_div, 0.0))
    logger.debug("js_distance: keys=%d, JSdiv=%.12g, JSdist=%.6f", len(keys), js_div, d)
    return d


def get_distance_fn(name: DistanceName) -> DistanceFn:
    if name == "tvd":
        return total_variation_distance
    if name == "hellinger":
        return hellinger_distance
    if name == "js":
        return js_distance
    raise ValueError(f"Unknown distance: {name}")


# ---------------------------------------------------------------------------
# Internal helpers for trajectories / snapshots
# ---------------------------------------------------------------------------

def _cumulative_freq_snapshots(
    algorithm: str,
    qubits: int,
    backend: str,
    batch: int,
    max_shots: int,
    seed: int = SEED,
) -> Tuple[List[Dict[str, int]], np.ndarray]:
    """
    Generate cumulative frequency snapshots at each batch boundary.
    """
    cumulative_freq: Dict[str, int] = {}
    snapshots: List[Dict[str, int]] = []
    shot_list: List[int] = []

    total_shots = 0
    batch_index = 0

    effective_max_shots = (max_shots // batch) * batch
    if effective_max_shots == 0:
        return snapshots, np.array([], dtype=int)

    while total_shots < effective_max_shots:
        batch_seed = seed + 1 + batch_index

        new_outcomes = get_outcomes(
            algorithm,
            qubits,
            backend,
            shots=batch,
            seed=batch_seed,
        )

        for outcome, count in new_outcomes.items():
            cumulative_freq[outcome] = cumulative_freq.get(outcome, 0) + int(count)

        total_shots += batch
        batch_index += 1

        snapshots.append(dict(cumulative_freq))
        shot_list.append(total_shots)

    shot_grid = np.asarray(shot_list, dtype=int)
    return snapshots, shot_grid


def _oracle_shots_from_distance(
    shot_grid: np.ndarray,
    dist_values: np.ndarray,
    epsilon: float,
) -> int:
    """
    Minimal shot count N such that dist(N') <= epsilon for all N' >= N,
    evaluated on the provided batch grid. Returns -1 if never satisfied.
    """
    if shot_grid.shape != dist_values.shape:
        raise ValueError("shot_grid and dist_values must have the same shape.")

    oracle_shots = -1
    suffix_ok = True

    for idx in range(len(dist_values) - 1, -1, -1):
        if dist_values[idx] > epsilon:
            suffix_ok = False
        elif suffix_ok:
            oracle_shots = int(shot_grid[idx])

    return oracle_shots


# Backward-compat alias (optional)
_oracle_shots_from_tvd = _oracle_shots_from_distance


# ---------------------------------------------------------------------------
# Analytic TVD-based shot forecasts (Hoeffding + union, Weissman-style)
# ---------------------------------------------------------------------------

def _forecast_shots_tvd_hoeffding_union(
    K: int,
    epsilon: float,
    confidence_level: float,
) -> float:
    """
    Hoeffding + union-bound forecast for TVD.
    """
    if K <= 0:
        raise ValueError("K must be a positive integer.")
    if not (0.0 < epsilon < 1.0):
        raise ValueError("epsilon must be in (0, 1).")
    if not (0.0 < confidence_level < 1.0):
        raise ValueError("confidence_level must be in (0, 1).")

    delta = 1.0 - confidence_level
    return (K**2 / (8.0 * epsilon**2)) * math.log((2.0 * K) / delta)


def _forecast_shots_tvd_weissman(
    K: int,
    epsilon: float,
    confidence_level: float,
) -> float:
    """
    Weissman-style L1/TVD deviation bound forecast for TVD.
    """
    if K <= 0:
        raise ValueError("K must be a positive integer.")
    if not (0.0 < epsilon < 1.0):
        raise ValueError("epsilon must be in (0, 1).")
    if not (0.0 < confidence_level < 1.0):
        raise ValueError("confidence_level must be in (0, 1).")

    delta = 1.0 - confidence_level
    log_term = K * math.log(2.0) + math.log(1.0 / delta)
    return log_term / (2.0 * epsilon**2)


def _tvd_epsilon_sufficient_for_distance(distance: DistanceName, eps: float) -> float:
    """
    Convert a target epsilon for 'distance' into a sufficient TVD epsilon
    (conservative) so we can reuse TVD analytic forecasts.

    - TVD: eps_tvd = eps
    - Hellinger: H^2 <= TVD => TVD <= eps^2 suffices for H <= eps
    - JS distance: JSD <= TVD and JS_dist = sqrt(JSD) => TVD <= eps^2 suffices
    """
    if distance == "tvd":
        return eps
    if distance in ("hellinger", "js"):
        return eps * eps
    raise ValueError(f"Unknown distance: {distance}")


def _forecast_shots_for_distance(
    distance: DistanceName,
    K: int,
    epsilon: float,
    confidence_level: float,
    batch: int,
) -> Tuple[int, int]:
    """
    Return (hoeffding_shots, weissman_shots) for the given distance.

    For non-TVD distances, uses a sufficient TVD condition (conservative).
    """
    eps_tvd = _tvd_epsilon_sufficient_for_distance(distance, float(epsilon))

    n_hoeff = _forecast_shots_tvd_hoeffding_union(
        K=K,
        epsilon=eps_tvd,
        confidence_level=confidence_level,
    )
    hoeffding_shots = int(math.ceil(n_hoeff / float(batch))) * batch

    n_weiss = _forecast_shots_tvd_weissman(
        K=K,
        epsilon=eps_tvd,
        confidence_level=confidence_level,
    )
    weissman_shots = int(math.ceil(n_weiss / float(batch))) * batch

    return hoeffding_shots, weissman_shots


# ---------------------------------------------------------------------------
# Multi-epsilon + multi-distance table (distance column + generic metrics)
# ---------------------------------------------------------------------------

def build_oracle_shot_table_multi_epsilon(
    epsilons: Sequence[float],
    batch: int = 50,
    max_shots: int = 20_000,
    use_final_as_reference: bool = True,
    confidence_level: float = 0.99,
    distances: Sequence[DistanceName] = ("tvd", "hellinger", "js"),
) -> pd.DataFrame:
    """
    For every epsilon, every (algorithm, qubits, backend), and every distance in
    `distances`, compute:

      - oracle_shots (empirical trajectory-based)
      - distance_reference_vs_gt
      - distance_oracle_vs_gt, distance_oracle_vs_reference
      - hoeffding_shots, weissman_shots (analytic; TVD-based and conservative for non-TVD)
      - distance_* metrics for those forecast shot counts (if within simulated range)

    Output includes a `distance` column.
    """
    eps_list = list(epsilons)
    if len(eps_list) == 0:
        raise ValueError("epsilons must be a non-empty sequence of floats.")
    for eps in eps_list:
        if not (0.0 < float(eps) < 1.0):
            raise ValueError(f"epsilon must be in (0, 1), got {eps}.")

    dist_list = list(distances)
    if len(dist_list) == 0:
        raise ValueError("distances must be a non-empty sequence.")
    for d in dist_list:
        get_distance_fn(d)  # validate

    if batch <= 0:
        raise ValueError("batch must be a positive integer.")
    if max_shots <= 0:
        raise ValueError("max_shots must be a positive integer.")
    if not (0.0 < confidence_level < 1.0):
        raise ValueError("confidence_level must be in (0, 1).")

    effective_max_shots = (max_shots // batch) * batch
    if effective_max_shots == 0:
        raise ValueError("max_shots must be at least as large as batch.")

    index = get_index()

    # ------------------------------------------------------------
    # Ground truth (GT) distributions: aer_simulator high-shot distro
    # ------------------------------------------------------------
    gts: Dict[Tuple[str, int], FreqCounts] = {}
    for alg, qubit_dict in index.items():
        for qubits, backends in qubit_dict.items():
            qubits_int = int(qubits)
            if "aer_simulator" in backends:
                gt_key = (alg, qubits_int)
                if gt_key not in gts:
                    gts[gt_key] = get_outcomes(
                        alg,
                        qubits_int,
                        "aer_simulator",
                        shots=REFERENCE_SHOTS,
                        seed=SEED,
                    )

    records: List[Tuple] = []

    for alg, qubit_dict in index.items():
        for qubits, backends in qubit_dict.items():
            qubits_int = int(qubits)
            gt_key = (alg, qubits_int)
            gt_freq = gts.get(gt_key, None)

            # Alphabet size for computational-basis outcomes
            K_states = 2 ** qubits_int

            for backend in backends:
                # 1) Build snapshots once for this triple.
                snapshots, shot_grid = _cumulative_freq_snapshots(
                    algorithm=alg,
                    qubits=qubits_int,
                    backend=backend,
                    batch=batch,
                    max_shots=effective_max_shots,
                    seed=SEED,
                )
                shot_grid_np = np.asarray(shot_grid, dtype=int)

                # 2) Choose reference distribution
                if not snapshots:
                    reference_freq: FreqCounts = {}
                else:
                    if not use_final_as_reference:
                        reference_seed = SEED + 1
                        reference_freq = get_outcomes(
                            alg,
                            qubits_int,
                            backend,
                            shots=REFERENCE_SHOTS,
                            seed=reference_seed,
                        )
                    else:
                        reference_freq = snapshots[-1]

                # 3) Run all distances
                for dist_name in dist_list:
                    dist_fn = get_distance_fn(dist_name)

                    # Trajectory against reference (one per distance)
                    if not snapshots:
                        dist_values = np.zeros_like(shot_grid_np, dtype=float)
                    else:
                        dist_values = np.asarray(
                            [float(dist_fn(freq, reference_freq)) for freq in snapshots],
                            dtype=float,
                        )

                    # distance(reference, GT) (independent of epsilon)
                    if gt_freq is not None and reference_freq:
                        distance_reference_vs_gt = float(dist_fn(reference_freq, gt_freq))
                    else:
                        distance_reference_vs_gt = float("nan")

                    # 4) Reuse trajectory for all epsilons
                    for eps in eps_list:
                        # Oracle
                        oracle_shots = _oracle_shots_from_distance(
                            shot_grid=shot_grid_np,
                            dist_values=dist_values,
                            epsilon=float(eps),
                        )

                        distance_oracle_vs_gt = float("nan")
                        distance_oracle_vs_reference = float("nan")

                        if snapshots and oracle_shots != -1:
                            idxs = np.where(shot_grid_np == oracle_shots)[0]
                            if len(idxs) > 0:
                                oidx = int(idxs[0])
                                oracle_freq = snapshots[oidx]
                                if reference_freq:
                                    distance_oracle_vs_reference = float(dist_fn(oracle_freq, reference_freq))
                                if gt_freq is not None:
                                    distance_oracle_vs_gt = float(dist_fn(oracle_freq, gt_freq))

                        # Forecasts (TVD-based; conservative for non-TVD)
                        try:
                            hoeffding_shots, weissman_shots = _forecast_shots_for_distance(
                                distance=dist_name,
                                K=K_states,
                                epsilon=float(eps),
                                confidence_level=confidence_level,
                                batch=batch,
                            )
                        except Exception:
                            hoeffding_shots, weissman_shots = -1, -1

                        # Evaluate forecast snapshots (if within simulated range)
                        distance_hoeffding_vs_gt = float("nan")
                        distance_hoeffding_vs_reference = float("nan")
                        if snapshots and hoeffding_shots != -1 and hoeffding_shots <= effective_max_shots:
                            idxs = np.where(shot_grid_np == hoeffding_shots)[0]
                            if len(idxs) > 0:
                                hidx = int(idxs[0])
                                freq_h = snapshots[hidx]
                                if reference_freq:
                                    distance_hoeffding_vs_reference = float(dist_fn(freq_h, reference_freq))
                                if gt_freq is not None:
                                    distance_hoeffding_vs_gt = float(dist_fn(freq_h, gt_freq))

                        distance_weissman_vs_gt = float("nan")
                        distance_weissman_vs_reference = float("nan")
                        if snapshots and weissman_shots != -1 and weissman_shots <= effective_max_shots:
                            idxs = np.where(shot_grid_np == weissman_shots)[0]
                            if len(idxs) > 0:
                                widx = int(idxs[0])
                                freq_w = snapshots[widx]
                                if reference_freq:
                                    distance_weissman_vs_reference = float(dist_fn(freq_w, reference_freq))
                                if gt_freq is not None:
                                    distance_weissman_vs_gt = float(dist_fn(freq_w, gt_freq))

                        records.append(
                            (
                                alg,
                                qubits_int,
                                backend,
                                dist_name,  # distance
                                float(eps),
                                int(batch),
                                int(effective_max_shots),
                                int(oracle_shots),
                                float(distance_reference_vs_gt),
                                float(distance_oracle_vs_gt),
                                float(distance_oracle_vs_reference),
                                int(hoeffding_shots),
                                float(distance_hoeffding_vs_gt),
                                float(distance_hoeffding_vs_reference),
                                int(weissman_shots),
                                float(distance_weissman_vs_gt),
                                float(distance_weissman_vs_reference),
                            )
                        )

    df = pd.DataFrame(
        records,
        columns=[
            "algorithm",
            "num_qubits",
            "backend",
            "distance",
            "epsilon",
            "batch",
            "max_shots",
            "oracle_shots",
            "distance_reference_vs_gt",
            "distance_oracle_vs_gt",
            "distance_oracle_vs_reference",
            "hoeffding_shots",
            "distance_hoeffding_vs_gt",
            "distance_hoeffding_vs_reference",
            "weissman_shots",
            "distance_weissman_vs_gt",
            "distance_weissman_vs_reference",
        ],
    )

    return df

In [ ]:
df = build_oracle_shot_table_multi_epsilon(
    epsilons=[0.01, 0.02, 0.05, 0.1, 0.2],)

df.to_csv("oracle_shot_table_multi_epsilon.csv", index=False)

df